In [1]:
# 3/8/19 - Database backend upgrade to cdac_db
%load_ext autoreload
%autoreload 2

In [2]:

def stripe_whitespaces_from_columnnames(df):
	new_columns = []
	columns = df.columns
	for column in columns:
		new_columns.append(column.rstrip())
	df.columns = new_columns
	return df


In [3]:
import pandas as pd
import numpy as np
import os
# dirdata='//mad3/MGH-NEURO-CDAC/Projects/CDAC_DATABASE/notebooks/workshop1p2/'

# filep=dirdata+'ICUSLEEP_DATA_2018-12-07_1344.csv'
# table=pd.read_csv(filep)
# print('status, enrolled:',len(table['csn']),'subjects')
# print('status, enrolled:',len(table['csn']),'csn')

In [4]:
# ICU SLEEP DATA:
## RedCap data:
RedCapExportDir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/'  
StudyDir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/Study/'

RedCapList = pd.read_csv(RedCapExportDir + 'LIST.csv')
CAM_Date = pd.read_csv(RedCapExportDir + 'CAM_Date.csv')
Consent_Date = pd.read_csv(RedCapExportDir + 'Consent_Date.csv')

# strip whitespaces from the right end of columnnames if there are some:
table =  stripe_whitespaces_from_columnnames(RedCapList)
CAM_Date =  stripe_whitespaces_from_columnnames(CAM_Date)
Consent_Date =  stripe_whitespaces_from_columnnames(Consent_Date)

In [5]:
STUDY_NAME='sedation'

In [6]:
table

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,ICU Site:,ICU Bed#:,MRN:,First Day Enrolled in Study:,First Name:,Last Name:
0,1,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,68,1298742.0,2018-06-06,Charles E,Sullivan
1,2,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Ellison 4,32,6378271.0,2018-06-11,Derek,Herling
2,3,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,54,1910753.0,2018-06-18,John,Carey
3,4,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,88,6188100.0,2018-06-19,Sheena,Foster
4,5,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 12,88,6051800.0,2018-06-19,Linda,Goodall
...,...,...,...,...,...,...,...,...,...,...
296,296,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,297,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,298,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,299,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
table[table['Study ID:'] == '8']

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,ICU Site:,ICU Bed#:,MRN:,First Day Enrolled in Study:,First Name:,Last Name:
7,8,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,88,2215049.0,2018-07-12,Diane E,Norris


In [8]:
# cannot locate subject 39: MRN = 843227
# 

In [8]:
df=pd.DataFrame(table,columns=['MRN:'])

In [9]:
mrns = list(set(df['MRN:'].dropna().apply(lambda x: int(x))))
mrns

[3603971,
 5761027,
 3718149,
 2792966,
 6774280,
 6689293,
 3724306,
 4953619,
 5553172,
 5216280,
 5858331,
 6615070,
 2092575,
 2771488,
 3750433,
 1104419,
 871462,
 2647079,
 6513707,
 6492716,
 3462701,
 6562863,
 4875316,
 1998902,
 6705207,
 6539323,
 1676863,
 1886786,
 6188100,
 6603851,
 3510348,
 2380363,
 1506383,
 6613073,
 6754388,
 1737814,
 2853463,
 684635,
 3969118,
 3876960,
 6195808,
 6563939,
 3165797,
 4905063,
 2596463,
 6572151,
 6673528,
 5946494,
 5503616,
 6577287,
 5407879,
 2215049,
 2086026,
 5420175,
 2523280,
 6590097,
 1420433,
 4295315,
 1810580,
 3468435,
 1849492,
 6450334,
 6119590,
 3708586,
 6545068,
 2264238,
 6313144,
 6693049,
 6676668,
 1649343,
 6095040,
 6572736,
 6223560,
 6289101,
 2249421,
 4840148,
 6443222,
 5789399,
 5190359,
 4101850,
 3826907,
 6280411,
 3159260,
 4297435,
 4523229,
 3042010,
 6777565,
 4105961,
 2770155,
 5094641,
 6587123,
 6175987,
 2021109,
 3591931,
 6479611,
 6407422,
 6436612,
 3223814,
 2010378,
 6782731,
 1

In [10]:
mrns_zeropadded = [str(x).zfill(7) for x in mrns if len(str(x))<7]
mrns_zeropadded

['0871462', '0684635', '0480086', '0843227']

In [11]:
mrn_s=",".join(list(map(lambda e: "'"+str(e)+"'", mrns))) + ',' + ",".join(list(map(lambda e: "'"+str(e)+"'", mrns_zeropadded)))
# 1876849 is changed MRN for subject 31 and subject 0871462 is zeropadded for subject 64, for both no adt entries have  been found., 021679 for studyid 139
mrn_s = mrn_s + ",'1876849'," + "'0021679'" + "'5307433'"  
# study id 39: csn 3235019411

In [12]:
csn_s = "'3234178903','3245471685','3251352836','3264388651','3264384407','3235019411'"

In [13]:
# add manual MRN, bedmaster time alignment check: 
# mrn_s = mrn_s +",'6644284'" +",'6663530'" +",'6677617'" +",'6662054'" + ",'6676303'" + ",'6674369'" + ",'6673721'"
mrn_s

"'3603971','5761027','3718149','2792966','6774280','6689293','3724306','4953619','5553172','5216280','5858331','6615070','2092575','2771488','3750433','1104419','871462','2647079','6513707','6492716','3462701','6562863','4875316','1998902','6705207','6539323','1676863','1886786','6188100','6603851','3510348','2380363','1506383','6613073','6754388','1737814','2853463','684635','3969118','3876960','6195808','6563939','3165797','4905063','2596463','6572151','6673528','5946494','5503616','6577287','5407879','2215049','2086026','5420175','2523280','6590097','1420433','4295315','1810580','3468435','1849492','6450334','6119590','3708586','6545068','2264238','6313144','6693049','6676668','1649343','6095040','6572736','6223560','6289101','2249421','4840148','6443222','5789399','5190359','4101850','3826907','6280411','3159260','4297435','4523229','3042010','6777565','4105961','2770155','5094641','6587123','6175987','2021109','3591931','6479611','6407422','6436612','3223814','2010378','6782731','

# GET BEDMASTER MONITOR

In [14]:
# get ADT monitor
import sys
import pandas as pd

### some parameters ###
#TODO: enter own username

user='ys858'
pwd=open('//mad3/MGH-NEURO-CDAC/Projects/CDAC_DATABASE/notebooks/utils/partners').read().strip()

module_p='//mad3/MGH-NEURO-CDAC/Projects/CDAC_DATABASE/notebooks/utils'
sys.path.append(module_p)
# sys.path.insert(0,module_p)
from common import connect_cdac,connect_edw, get_module_version

#TODO: get version num? connection string message?
print(get_module_version())

# conn1=connect_edw(user,pwd)
cxn=connect_cdac()


0.2
CDAC connection <pyodbc.Connection object at 0x0000029946170990>


In [15]:
sql="""
  SELECT cdac_latest_udpate=max(CDACModifiedTime) FROM [cdac_db].[dbo].[vw_Bedmaster]
"""
pd.read_sql_query(sql,cxn)

,cdac_latest_udpate
0,2020-03-31 09:38:31.690


In [16]:
# 
# This mechanism makes it feasible to quickly scan for recent ADT
# Once identified it shows it
#

### Clearning ADT buffer and refilling ###
def sql_clear_adt():
    cur = cxn.cursor()
    sql="""
    delete from tblADT_sedation2
    """   
    try:    
        cur.execute(sql)
        cxn.commit()

    except:
        print("update ADT buffer failed")    
    cur.close()
    print('===> clearing ADT finished')

def sql_update_adt():
#     m_s=str(MONITOR_TIME)
    
    cur = cxn.cursor()
    sql="""
     
   insert into tblADT_sedation2
   select * from cdac_db.dbo.ADT_Extended
          WHERE (1=1)
       AND (MRN in (%s))
       OR (PatientEncounterID in (%s))

    """ % (mrn_s, csn_s)
    
    print(sql)
        
    try:
        cur.execute(sql)
        cxn.commit()

    except:
        print("update ADT buffer failed")
    
    cur.close()        
    print('===> updating ADT finished')
        

In [24]:
# ADT,
if 1:
    sql_clear_adt()
    sql_update_adt()

===> clearing ADT finished

     
   insert into tblADT_sedation2
   select * from cdac_db.dbo.ADT_Extended
          WHERE (1=1)
       AND (MRN in ('3603971','5761027','3718149','2792966','6774280','6689293','3724306','4953619','5553172','5216280','5858331','6615070','2092575','2771488','3750433','1104419','871462','2647079','6513707','6492716','3462701','6562863','4875316','1998902','6705207','6539323','1676863','1886786','6188100','6603851','3510348','2380363','1506383','6613073','6754388','1737814','2853463','684635','3969118','3876960','6195808','6563939','3165797','4905063','2596463','6572151','6673528','5946494','5503616','6577287','5407879','2215049','2086026','5420175','2523280','6590097','1420433','4295315','1810580','3468435','1849492','6450334','6119590','3708586','6545068','2264238','6313144','6693049','6676668','1649343','6095040','6572736','6223560','6289101','2249421','4840148','6443222','5789399','5190359','4101850','3826907','6280411','3159260','4297435','4523229','3

In [25]:

sql="""
select * from tblADT_sedation2
"""

df_d=pd.read_sql_query(sql,cxn)

print(df_d)
# print('recent records',len(df_d),df_d.sort_values(by='StartTime').head(1)['StartTime'].values,df_d.sort_values(by='StartTime').tail(1)['StartTime'].values)

          MRN PatientEncounterID InEventID           InEventDSC OutEventID  \
0     3591931         3241454794  59392644          Transfer In   59392657   
1     3591931         3241454794  59392658          Transfer In   59394109   
2     3591931         3241454794  59496302          Transfer In   59537465   
3     2853463         3244602566  60448129  Hospital Outpatient   60483080   
4     1737814         3243386495  59950401  Hospital Outpatient   59973335   
...       ...                ...       ...                  ...        ...   
9213  6280411         3278692640  71258103  Hospital Outpatient   71554689   
9214  5854072         3279729264  71612551  Hospital Outpatient   71922437   
9215  5854072         3279643429  71550406  Hospital Outpatient   71922444   
9216  5854072         3279643266  71550362  Hospital Outpatient   71922448   
9217  5854072         3279643239  71550356  Hospital Outpatient   71922449   

       OutEventDSC Hospital PatientClassDSC DepartmentID  \
0  

In [22]:
#### Using EDW backend
# # vw_Bedmaster_ADT is the ICU only monitoring that pairs it to bedmaster table
# else:
print('===> scanning ADT monitor buffer')
sql="""
SELECT       *
FROM cdac_db.dbo.vw_Bedmaster_ADT_sedation
WHERE        (1 = 1) /* within 90 days*/
and
(MRN in (%s)) or (PatientEncounterID in (%s))
""" % (mrn_s, csn_s)


df_adt0=pd.read_sql_query(sql,cxn)

===> scanning ADT monitor buffer


#### save ADT information, will be added to ICU sleep data table in ICUSleep_DataTable.ipynb file

In [21]:
df_adt0.to_csv(os.path.join(StudyDir,'ADT_cdacdb.csv'), index=False)

In [22]:
df_adt0.head()

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
0,4736799,1.562135e+09,1.563044e+09,1562163540,1.562703e+09,B1274,2019-07-03 09:19:00.0000000,2019-07-09 15:08:00.0000000,3258769573,MGH BLAKE 12 ICU,B1274 A,2019-07-03 01:17:00.0000000,2019-07-13 13:59:00.0000000
1,2390500,1.560536e+09,1.562627e+09,1561303980,1.561560e+09,E402,2019-06-23 10:33:00.0000000,2019-06-26 09:34:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
2,2390500,1.560536e+09,1.562627e+09,1562009160,1.562451e+09,E402,2019-07-01 14:26:00.0000000,2019-07-06 17:08:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
3,2390500,1.560536e+09,1.562627e+09,1561574460,1.562000e+09,E402,2019-06-26 13:41:00.0000000,2019-07-01 11:54:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
4,2372026,1.548968e+09,1.550347e+09,1549061820,1.550347e+09,G936,2019-02-01 17:57:00.0000000,2019-02-16 15:00:00.0000000,3235976982,MGH BIGELOW 9 MED,G0936 A,2019-01-31 15:53:00.0000000,2019-02-16 15:00:00.0000000


In [26]:
df_adt0[df_adt0.MRN == '5307433']

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS


In [23]:
df_adt0[df_adt0.MRN == '6547241'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS


In [24]:
df_adt0[df_adt0.MRN == '1910753']

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
192,1910753,1.484753e+09,1.485464e+09,1484777460,1.485464e+09,W1030,2017-01-18 17:11:00.0000000,2017-01-26 15:46:00.0000000,3142989191,MGH WHITE 10 MEDICINE,W1030 B,2017-01-18 10:21:00.0000000,2017-01-26 15:46:00.0000000
296,1910753,1.529270e+09,1.530216e+09,1529302320,1.529636e+09,B754,2018-06-18 01:12:00.0000000,2018-06-21 21:56:00.0000000,3205625962,MGH BLAKE 7 MICU,B0754 A,2018-06-17 16:07:00.0000000,2018-06-28 14:53:00.0000000


In [25]:
df_adt0.tail()

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
747,5854072,1.563171e+09,1.565900e+09,1563388980,1.563572e+09,B1278,2019-07-17 13:43:00.0000000,2019-07-19 16:41:00.0000000,3260213550,MGH BLAKE 12 ICU,B1278 A,2019-07-15 01:13:00.0000000,2019-08-15 15:21:00.0000000
748,5854072,1.563171e+09,1.565900e+09,1563572460,1.565278e+09,G944,2019-07-19 16:41:00.0000000,2019-08-08 10:30:00.0000000,3260213550,MGH BIGELOW 9 MED,G0944 A,2019-07-15 01:13:00.0000000,2019-08-15 15:21:00.0000000
749,5854072,1.563171e+09,1.565900e+09,1565304180,1.565900e+09,G944,2019-08-08 17:43:00.0000000,2019-08-15 15:21:00.0000000,3260213550,MGH BIGELOW 9 MED,G0944 A,2019-07-15 01:13:00.0000000,2019-08-15 15:21:00.0000000
750,1750989,1.574766e+09,NaN,1575747300,1.576038e+09,B1260,2019-12-07 14:35:00.0000000,2019-12-10 23:19:00.0000000,3278402307,MGH BLAKE 12 ICU,B1260 A,2019-11-26 05:56:00.0000000,2020-01-01 00:00:00.0000000
751,1750989,1.574766e+09,NaN,1574825460,1.575747e+09,B1272,2019-11-26 22:31:00.0000000,2019-12-07 14:35:00.0000000,3278402307,MGH BLAKE 12 ICU,B1272 A,2019-11-26 05:56:00.0000000,2020-01-01 00:00:00.0000000


In [26]:
### precondition - exit if adt is empty
df_adt=df_adt0
df_adt.dropna(subset=['MRN'],inplace=True)

if len(df_adt) == 0:
    print('===> EXIT:','no MRN found')
    exit()
else:
    print('===> CONTINUE:','MRN found')

===> CONTINUE: MRN found


In [27]:
df_adt.head()

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
0,4736799,1.562135e+09,1.563044e+09,1562163540,1.562703e+09,B1274,2019-07-03 09:19:00.0000000,2019-07-09 15:08:00.0000000,3258769573,MGH BLAKE 12 ICU,B1274 A,2019-07-03 01:17:00.0000000,2019-07-13 13:59:00.0000000
1,2390500,1.560536e+09,1.562627e+09,1561303980,1.561560e+09,E402,2019-06-23 10:33:00.0000000,2019-06-26 09:34:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
2,2390500,1.560536e+09,1.562627e+09,1562009160,1.562451e+09,E402,2019-07-01 14:26:00.0000000,2019-07-06 17:08:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
3,2390500,1.560536e+09,1.562627e+09,1561574460,1.562000e+09,E402,2019-06-26 13:41:00.0000000,2019-07-01 11:54:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
4,2372026,1.548968e+09,1.550347e+09,1549061820,1.550347e+09,G936,2019-02-01 17:57:00.0000000,2019-02-16 15:00:00.0000000,3235976982,MGH BIGELOW 9 MED,G0936 A,2019-01-31 15:53:00.0000000,2019-02-16 15:00:00.0000000


In [28]:
print('num of adt records',len(df_adt))
print('num of mrns',len(df_adt.groupby('MRN')))
print('num of csns',len(df_adt.groupby('PatientEncounterID')))
# print(df_adt)    


num of adt records 752
num of mrns 195
num of csns 384


In [29]:
df_visitg=df_adt.groupby(['MRN'])

def reindex_by_date(df):
    dates = pd.date_range(df.index.min(), df.index.max())
    return df.reindex(dates).ffill()



In [30]:
df_adt.head()

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
0,4736799,1.562135e+09,1.563044e+09,1562163540,1.562703e+09,B1274,2019-07-03 09:19:00.0000000,2019-07-09 15:08:00.0000000,3258769573,MGH BLAKE 12 ICU,B1274 A,2019-07-03 01:17:00.0000000,2019-07-13 13:59:00.0000000
1,2390500,1.560536e+09,1.562627e+09,1561303980,1.561560e+09,E402,2019-06-23 10:33:00.0000000,2019-06-26 09:34:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
2,2390500,1.560536e+09,1.562627e+09,1562009160,1.562451e+09,E402,2019-07-01 14:26:00.0000000,2019-07-06 17:08:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
3,2390500,1.560536e+09,1.562627e+09,1561574460,1.562000e+09,E402,2019-06-26 13:41:00.0000000,2019-07-01 11:54:00.0000000,3256083493,MGH ELLISON 4 SICU,E0402 A,2019-06-14 13:20:00.0000000,2019-07-08 17:56:00.0000000
4,2372026,1.548968e+09,1.550347e+09,1549061820,1.550347e+09,G936,2019-02-01 17:57:00.0000000,2019-02-16 15:00:00.0000000,3235976982,MGH BIGELOW 9 MED,G0936 A,2019-01-31 15:53:00.0000000,2019-02-16 15:00:00.0000000


In [27]:
#### check ADT for CSN
sql="""
SELECT       *
FROM cdac_db.dbo.vw_Bedmaster_ADT_sedation
WHERE        (1 = 1) /* within 90 days*/
and
(MRN in (%s)) or (PatientEncounterID in (%s))
""" % (mrn_s, csn_s)


df_adt0=pd.read_sql_query(sql,cxn)

##### subjectIDs for which no bedmaster file was found yet:

In [32]:
#subject 31 changed mrn:
df_adt0[df_adt0.MRN == '021679'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS


In [28]:
df_adt0[df_adt0.MRN == '5307433'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS


In [33]:
df_adt0[df_adt0.PatientEncounterID == '3264388651']

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
418,6728191,1.565583e+09,1.566599e+09,1565830140,1.566171e+09,B1254,2019-08-14 19:49:00.0000000,2019-08-18 18:23:00.0000000,3264388651,MGH BLAKE 12 ICU,B1254 A,2019-08-11 23:03:00.0000000,2019-08-23 17:19:00.0000000


In [34]:
# subject 64:
df_adt0[df_adt0.MRN == '6728191'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
418,6728191,1.565583e+09,1.566599e+09,1565830140,1.566171e+09,B1254,2019-08-14 19:49:00.0000000,2019-08-18 18:23:00.0000000,3264388651,MGH BLAKE 12 ICU,B1254 A,2019-08-11 23:03:00.0000000,2019-08-23 17:19:00.0000000


In [35]:
df_adt0[df_adt0.PatientEncounterID == '3245471685']

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
419,0871462,1.554405e+09,1.555618e+09,1554449820,1.554495e+09,B760,2019-04-05 02:37:00.0000000,2019-04-05 15:06:00.0000000,3245471685,MGH BLAKE 7 MICU,B0760 A,2019-04-04 14:18:00.0000000,2019-04-18 15:05:00.0000000
505,0871462,1.554405e+09,1.555618e+09,1554557580,1.555027e+09,B1274,2019-04-06 08:33:00.0000000,2019-04-11 19:02:00.0000000,3245471685,MGH BLAKE 12 ICU,B1274 A,2019-04-04 14:18:00.0000000,2019-04-18 15:05:00.0000000


In [36]:
#subject 64 zero-padded
df_adt0[df_adt0.MRN == '0871462'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
419,0871462,1.554405e+09,1.555618e+09,1554449820,1.554495e+09,B760,2019-04-05 02:37:00.0000000,2019-04-05 15:06:00.0000000,3245471685,MGH BLAKE 7 MICU,B0760 A,2019-04-04 14:18:00.0000000,2019-04-18 15:05:00.0000000
505,0871462,1.554405e+09,1.555618e+09,1554557580,1.555027e+09,B1274,2019-04-06 08:33:00.0000000,2019-04-11 19:02:00.0000000,3245471685,MGH BLAKE 12 ICU,B1274 A,2019-04-04 14:18:00.0000000,2019-04-18 15:05:00.0000000


In [37]:
df_adt0[df_adt0.PatientEncounterID == '3264388651']

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
418,6728191,1.565583e+09,1.566599e+09,1565830140,1.566171e+09,B1254,2019-08-14 19:49:00.0000000,2019-08-18 18:23:00.0000000,3264388651,MGH BLAKE 12 ICU,B1254 A,2019-08-11 23:03:00.0000000,2019-08-23 17:19:00.0000000


In [38]:
#subject 87
df_adt0[df_adt0.MRN == '3462701'] 

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
324,3462701,1.557867e+09,1.560194e+09,1557891600,1.557958e+09,W1022,2019-05-14 22:40:00.0000000,2019-05-15 17:01:00.0000000,3251352836,MGH WHITE 10 MEDICINE,W1022 B,2019-05-14 15:52:00.0000000,2019-06-10 14:20:00.0000000
325,3462701,1.557867e+09,1.560194e+09,1557957660,1.559174e+09,W1012,2019-05-15 17:01:00.0000000,2019-05-29 18:45:00.0000000,3251352836,MGH WHITE 10 MEDICINE,W1012 B,2019-05-14 15:52:00.0000000,2019-06-10 14:20:00.0000000
326,3462701,1.557867e+09,1.560194e+09,1559204700,1.559317e+09,B1278,2019-05-30 03:25:00.0000000,2019-05-31 10:43:00.0000000,3251352836,MGH BLAKE 12 ICU,B1278 A,2019-05-14 15:52:00.0000000,2019-06-10 14:20:00.0000000
327,3462701,1.557867e+09,1.560194e+09,1559321040,1.559696e+09,B1278,2019-05-31 11:44:00.0000000,2019-06-04 20:00:00.0000000,3251352836,MGH BLAKE 12 ICU,B1278 A,2019-05-14 15:52:00.0000000,2019-06-10 14:20:00.0000000
441,3462701,1.500614e+09,1.501630e+09,1500614100,1.500849e+09,B780,2017-07-21 00:15:00.0000000,2017-07-23 17:35:00.0000000,3164479878,MGH BLAKE 7 MICU,B0780 A,2017-07-21 00:15:00.0000000,2017-08-01 18:27:00.0000000


In [39]:
# Get bedmaster - query1 - num, query2 - str
sql_var={}
sql_var['mrn_s']=",".join(["'"+str(e) +"'" for e in df_adt['MRN'].map(str).unique().tolist()])

In [40]:
sql_var['mrn_s']

"'4736799','2390500','2372026','4953619','3591931','1835922','4599114','2215049','1420433','4875316','5858331','6223560','1480537','5338435','5946494','6420799','5101053','2264238','1737814','6407422','1298742','6254435','1902538','6754388','3592022','2523280','1788840','1933710','3165797','5190359','6612945','4523229','1990504','4352857','1469338','6562863','6248867','2853463','3759088','4877206','6513707','5854072','1723880','6436612','6782731','0480086','4981686','5761027','2010378','6777565','3826907','3243449','6774280','2092575','2792966','5420175','1676863','3223814','6188100','4295315','2865943','6221251','4800952','6177186','2881848','6292253','6492716','4098905','5553172','1876849','6378271','1983461','1742644','6499801','4840148','6590097','6413586','1910753','1486829','2898315','6546904','6479611','1909198','6313144','6289101','3468435','2630974','3510348','6676668','3718149','2194847','5503616','2770155','6113072','5789399','2771488','4811749','1975166','6731074','6195808'

In [41]:
### Refulling tblPatientBedmaster for sedation


def sql_clear_pbm():
#     sql_var={}

    cur = cxn.cursor()
    ### remove old data
    ### commented part: if you want to remove specified MRNs only, active/decommented: remove all mrns for update.
#     sql="""
#     delete from tblPatientBedmaster_""" + STUDY_NAME + """ where MRN in (%s)
# """ % (mrn_s)
    sql="""
    delete from tblPatientBedmaster_""" + STUDY_NAME + ""    
    print(sql)

    # cxn.close()    
    try:    
        cur.execute(sql)
        cxn.commit()

    except Exception as e:
        print("update PBM buffer failed",e)
    cur.close()
    print('==> clear PBM finished')
    
def sql_update_pbm():
    cur = cxn.cursor()
    #### insert new data
    sql = """
with vw_id as (
select * from cdac_db.dbo.vw_Bedmaster_ADT_sedation
)
,vw_pbm as (
SELECT        TransferInDTS, TransferOutDTS, a0.stpLoc as stpIn, tStart, tEnd, dbo.DateTimeToUNIXLocal(a0.DischargeDTS) AS tDischarge, dbo.DateTimeToUNIXLocal(a0.AdmissionDTS) AS tAdmit, fileID,unixFileStartTime, unixFileEndTime, fileStartTime,
                         a0.[MRN] AS MRN, a0.PatientEncounterID, concat(a0.stpLoc, '_', tStart) AS grp, concat(a0.stpLoc, '_', tStart,'_',unixFileStartTime) AS ugid
FROM             vw_id a0 						 
						 CROSS apply dbo.fn_get_bm_fileID(a0.stpLoc, a0.tStart, a0.tEnd, dbo.DateTimeToUNIXLocal(a0.DischargeDTS))
)

insert into tblPatientBedmaster_sedation
select * from vw_pbm
    """
    
#     print(sql_var)
    
#     print(sql)

    cur = cxn.cursor()
    try:
        cur.execute(sql)
        cxn.commit()
    
    except Exception as e:
        print("update PBM buffer failed",e)
        
    cur.close()
    print('===> update PBM finished')


In [42]:
# cur = cxn.cursor()
# ### remove old data
# sql="""
# delete from tblPatientBedmaster_sedation
# """ 

# print(sql)

# # cxn.close()    
# try:    
#     cur.execute(sql)
#     cxn.commit()

# except Exception as e:
#     print("update PBM buffer failed",e)
# cur.close()
# print('==> clear PBM finished')

In [43]:
# This run the bedmaster linking - (NEW VERSION that works with cdac_db.dbo.ADT)
if 0:
    sql_clear_pbm()
    sql_update_pbm()

In [44]:

# Get bedmaster linking

if len(df_adt) > 0:
    host_p='//mad3.partners.org/MGH-NEURO-CDAC'
#     sql="select * from tblPatientBedmaster_sedation where VisitIdentifier in (%s)" % sql_var['visit_id_s']
    sql="select * from tblPatientBedmaster_sedation"
    print(sql)
    df_pbm0=pd.read_sql_query(sql,cxn)
    print('matching found',len(df_pbm0.groupby('MRN')))
    print(df_pbm0)



select * from tblPatientBedmaster_sedation
matching found 195
               StartTime             EndTime  stpIn      tStart          tEnd  \
0    2019-07-03 09:19:00 2019-07-09 15:08:00  B1274  1562163540  1.562703e+09   
1    2019-07-03 09:19:00 2019-07-09 15:08:00  B1274  1562163540  1.562703e+09   
2    2019-06-23 10:33:00 2019-06-26 09:34:00   E402  1561303980  1.561560e+09   
3    2019-06-23 10:33:00 2019-06-26 09:34:00   E402  1561303980  1.561560e+09   
4    2019-07-01 14:26:00 2019-07-06 17:08:00   E402  1562009160  1.562451e+09   
...                  ...                 ...    ...         ...           ...   
1024 2019-06-07 18:12:00 2019-06-17 15:18:00  L1092  1559949120  1.560803e+09   
1025 2018-11-05 20:36:00 2018-11-06 21:03:00   B876  1541468160  1.541556e+09   
1026 2018-11-11 11:23:00 2018-11-11 14:56:00   B884  1541953380  1.541966e+09   
1027 2018-11-11 11:23:00 2018-11-11 14:56:00   B884  1541953380  1.541966e+09   
1028 2019-06-28 00:04:00 2019-06-28 15:13:00  B

In [45]:
print(df_pbm0[df_pbm0.fileID == 'ELL04_438-1539459164'])

             StartTime             EndTime stpIn      tStart          tEnd  \
46 2018-10-13 15:19:00 2018-10-16 17:36:00  E438  1539461940  1.539729e+09   

    tDischarge        tAdmit                fileID  unixFileStartTime  \
46  1540412160  1.539341e+09  ELL04_438-1539459164         1539459164   

    unixFileEndTime       fileStartTime      MRN VisitIdentifier  \
46       1539742974 2018-10-13 19:32:44  1902538      3218542920   

                grp                        ugid  
46  E438_1539461940  E438_1539461940_1539459164  


In [46]:
print(df_pbm0[df_pbm0.fileID == 'BLK12_1252-1549081788'])

              StartTime             EndTime  stpIn      tStart          tEnd  \
879 2019-02-12 18:34:00 2019-02-14 15:59:00  B1252  1550014440  1.550178e+09   

     tDischarge        tAdmit                 fileID  unixFileStartTime  \
879  1550428140  1.549968e+09  BLK12_1252-1549081788         1549081788   

     unixFileEndTime       fileStartTime      MRN VisitIdentifier  \
879       1550015346 2019-02-02 04:29:48  0843227      3235019411   

                  grp                         ugid  
879  B1252_1550014440  B1252_1550014440_1549081788  


In [47]:
# study id 3
df_pbm0[df_pbm0.MRN == '1910753']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
858,2018-06-18 01:12:00,2018-06-21 21:56:00,B754,1529302320,1.529636e+09,1530215580,1.529270e+09,MICU_754-1447035332,1447035332,1577854800,2015-11-09 02:15:32,1910753,3205625962,B754_1529302320,B754_1529302320_1447035332
859,2018-06-18 01:12:00,2018-06-21 21:56:00,B754,1529302320,1.529636e+09,1530215580,1.529270e+09,BLK07_754-1529299617,1529299617,1529685114,2018-06-18 05:26:57,1910753,3205625962,B754_1529302320,B754_1529302320_1529299617


In [48]:
# studyid 8
df_pbm0[df_pbm0.MRN == '2215049']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
54,2016-01-22 13:26:00,2016-01-29 14:50:00,B786,1453487160,1.454097e+09,1454097000,1.453129e+09,BLK07_786-1450737062,1450737062,1453488744,2015-12-21 22:31:02,2215049,3104671046,B786_1453487160,B786_1453487160_1450737062
55,2016-01-22 13:26:00,2016-01-29 14:50:00,B786,1453487160,1.454097e+09,1454097000,1.453129e+09,BLK07_786-1453488744,1453488744,1454111521,2016-01-22 18:52:24,2215049,3104671046,B786_1453487160,B786_1453487160_1453488744
56,2016-01-22 13:26:00,2016-01-29 14:50:00,B786,1453487160,1.454097e+09,1454097000,1.453129e+09,MICU_786-1446819364,1446819364,1577854800,2015-11-06 14:16:04,2215049,3104671046,B786_1453487160,B786_1453487160_1446819364
449,2018-10-26 00:17:00,2018-10-30 06:26:00,B770,1540531020,1.540899e+09,1541615580,1.540497e+09,BLK07_770-1540894334,1540894334,1540926895,2018-10-30 10:12:14,2215049,3222970454,B770_1540531020,B770_1540531020_1540894334
450,2018-10-26 00:17:00,2018-10-30 06:26:00,B770,1540531020,1.540899e+09,1541615580,1.540497e+09,MICU_770-1447076816,1447076816,1577854800,2015-11-09 13:46:56,2215049,3222970454,B770_1540531020,B770_1540531020_1447076816
451,2018-10-26 00:17:00,2018-10-30 06:26:00,B770,1540531020,1.540899e+09,1541615580,1.540497e+09,BLK07_770-1540528007,1540528007,1540894334,2018-10-26 04:26:47,2215049,3222970454,B770_1540531020,B770_1540531020_1540528007
798,2018-07-11 18:07:00,2018-07-15 22:08:00,B788,1531350420,1.531710e+09,1532642400,1.531270e+09,BLK07_788-1531408226,1531408226,1531734139,2018-07-12 15:10:26,2215049,3208824162,B788_1531350420,B788_1531350420_1531408226
799,2018-07-11 18:07:00,2018-07-15 22:08:00,B788,1531350420,1.531710e+09,1532642400,1.531270e+09,MICU_788-1446453804,1446453804,1577854800,2015-11-02 08:43:24,2215049,3208824162,B788_1531350420,B788_1531350420_1446453804
800,2018-07-11 18:07:00,2018-07-15 22:08:00,B788,1531350420,1.531710e+09,1532642400,1.531270e+09,BLK07_788-1531346646,1531346646,1531408226,2018-07-11 22:04:06,2215049,3208824162,B788_1531350420,B788_1531350420_1531346646
801,2017-03-19 17:14:00,2017-03-25 22:59:00,E404,1489961640,1.490501e+09,1490727960,1.489743e+09,ELL04_404-1489957902,1489957902,1490505686,2017-03-19 21:11:42,2215049,3149911223,E404_1489961640,E404_1489961640_1489957902


In [49]:
# studyid 15
df_pbm0[df_pbm0.MRN == '5553172']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
99,2018-10-19 17:39:00,2018-10-31 10:00:00,G920,1539988740,1.540998e+09,1540998000,1.539107e+09,BIG9_920-1499577741,1499577741,1552680907,2017-07-09 05:22:21,5553172,3220601732,G920_1539988740,G920_1539988740_1499577741
399,2018-10-09 21:00:00,2018-10-15 18:48:00,B758,1539136800,1.539647e+09,1540998000,1.539107e+09,MICU_758-1445357903,1445357903,1577854800,2015-10-20 16:18:23,5553172,3220601732,B758_1539136800,B758_1539136800_1445357903
400,2018-10-09 21:00:00,2018-10-15 18:48:00,B758,1539136800,1.539647e+09,1540998000,1.539107e+09,BLK07_758-1539133548,1539133548,1539647987,2018-10-10 01:05:48,5553172,3220601732,B758_1539136800,B758_1539136800_1539133548
401,2018-10-15 18:48:00,2018-10-19 16:11:00,G920,1539647280,1.539983e+09,1540998000,1.539107e+09,BIG9_920-1499577741,1499577741,1552680907,2017-07-09 05:22:21,5553172,3220601732,G920_1539647280,G920_1539647280_1499577741


In [50]:
# studyid 16
df_pbm0[df_pbm0.MRN == '1902538']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
45,2018-10-13 15:19:00,2018-10-16 17:36:00,E438,1539461940,1.539729e+09,1540412160,1.539341e+09,ELL4_438-1445901518,1445901518,1577854800,2015-10-26 23:18:38,1902538,3218542920,E438_1539461940,E438_1539461940_1445901518
46,2018-10-13 15:19:00,2018-10-16 17:36:00,E438,1539461940,1.539729e+09,1540412160,1.539341e+09,ELL04_438-1539459164,1539459164,1539742974,2018-10-13 19:32:44,1902538,3218542920,E438_1539461940,E438_1539461940_1539459164


In [51]:
# studyid 23
df_pbm0[df_pbm0.MRN == '6413586']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
482,2019-01-11 17:23:00,2019-01-16 15:59:00,E434,1547245380,1.547672e+09,1547672340,1.547149e+09,ELL04_434-1547245531,1547245531,1547678133,2019-01-11 22:25:31,6413586,3226252684,E434_1547245380,E434_1547245380_1547245531
483,2019-01-11 17:23:00,2019-01-16 15:59:00,E434,1547245380,1.547672e+09,1547672340,1.547149e+09,ELL4_434-1445305762,1445305762,1577854800,2015-10-20 01:49:22,6413586,3226252684,E434_1547245380,E434_1547245380_1445305762
484,2019-01-11 17:23:00,2019-01-16 15:59:00,E434,1547245380,1.547672e+09,1547672340,1.547149e+09,ELL04_434-1547056025,1547056025,1547245531,2019-01-09 17:47:05,6413586,3226252684,E434_1547245380,E434_1547245380_1547056025
781,2018-09-11 14:05:00,2018-09-13 14:30:00,E434,1536692700,1.536867e+09,1537225380,1.536614e+09,ELL4_434-1445305762,1445305762,1577854800,2015-10-20 01:49:22,6413586,3211677557,E434_1536692700,E434_1536692700_1445305762
782,2018-09-11 14:05:00,2018-09-13 14:30:00,E434,1536692700,1.536867e+09,1537225380,1.536614e+09,ELL04_434-1536686163,1536686163,1536869846,2018-09-11 17:16:03,6413586,3211677557,E434_1536692700,E434_1536692700_1536686163


In [52]:
# studyid 25
df_pbm0[df_pbm0.MRN == '5503616']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
142,2018-12-03 18:28:00,2019-01-08 03:00:00,B876,1543879680,1.546934e+09,1546934400,1.543833e+09,BLK08_876-1543880125,1543880125,1547007200,2018-12-03 23:35:25,5503616,3220127455,B876_1543879680,B876_1543879680_1543880125
143,2018-12-03 18:28:00,2019-01-08 03:00:00,B876,1543879680,1.546934e+09,1546934400,1.543833e+09,BLK08_876-1543782970,1543782970,1543880125,2018-12-02 20:36:10,5503616,3220127455,B876_1543879680,B876_1543879680_1543782970
702,2019-01-14 22:56:00,2019-01-18 13:25:00,B1254,1547524560,1.547836e+09,1547835900,1.547492e+09,BLK12_1254-1547520918,1547520918,1547845366,2019-01-15 02:55:18,5503616,3233147983,B1254_1547524560,B1254_1547524560_1547520918


In [49]:
df_pbm=df_pbm0

# DROP ALL ENTRIES BEFORE APRIL 2018:
df_pbm = df_pbm.mask(df_pbm.StartTime<pd.Timestamp(2018,4,1)).dropna(how='all')
df_pbm = df_pbm.mask(df_pbm.fileStartTime<pd.Timestamp(2018,4,1)).dropna(how='all')

print(len(set(df_pbm['fileID'])))


534


In [50]:
df_pbm[df_pbm.MRN == '1876849']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
404,2018-12-30 00:50:00,2019-01-02 18:29:00,E924,1.546149e+09,1.546472e+09,1.546986e+09,1.546131e+09,ELL09_924-1546145882,1.546146e+09,1.546478e+09,2018-12-30 04:58:02,1876849,3231166228,E924_1546149000,E924_1546149000_1546145882
593,2019-01-22 18:29:00,2019-01-26 18:03:00,B756,1.548200e+09,1.548544e+09,1.549489e+09,1.548178e+09,BLK07_756-1547899217,1.547899e+09,1.548200e+09,2019-01-19 12:00:17,1876849,3234178903,B756_1548199740,B756_1548199740_1547899217
594,2019-01-22 18:29:00,2019-01-26 18:03:00,B756,1.548200e+09,1.548544e+09,1.549489e+09,1.548178e+09,BLK07_756-1548200185,1.548200e+09,1.548200e+09,2019-01-22 23:36:25,1876849,3234178903,B756_1548199740,B756_1548199740_1548200185
595,2019-01-22 18:29:00,2019-01-26 18:03:00,B756,1.548200e+09,1.548544e+09,1.549489e+09,1.548178e+09,BLK07_756-1548200241,1.548200e+09,1.548569e+09,2019-01-22 23:37:21,1876849,3234178903,B756_1548199740,B756_1548199740_1548200241


In [51]:
file_id_s=",".join(list(map(lambda e: "'"+str(e)+"'", list(set(df_pbm['fileID'])))))

In [52]:
### Check files exist
sql="""
select * from vw_BedmasterMat where
fileID in (%s)
""" % file_id_s


df_files_mat=pd.read_sql_query(sql,cxn)

In [53]:
print('fileIDs converted','|','filesID found')
print(len(df_files_mat.groupby('fileID')),'|',len(set(df_pbm['fileID'])))

fileIDs converted | filesID found
244 | 534


In [54]:
# 198 223

In [55]:
df_files_mat[df_files_mat.fileID == 'ELL04_402-1549599295']

,CreationDTS,size,fileName,path,seg,fileID,unixStartTime,est_seg_start,segStartTime,CDACModifiedTime
1015,20190618.123049.5001162000,302782296,ELL04_402-1549599295_0_fixed.mat,2019/2/ELL04_402-1549599295_0_fixed.mat,0,ELL04_402-1549599295,1549599295,1549599295,2019-02-07 23:14:55,2019-12-26 03:01:04.727
1016,20190618.123105.4369555000,269631230,ELL04_402-1549599295_1_fixed.mat,2019/2/ELL04_402-1549599295_1_fixed.mat,1,ELL04_402-1549599295,1549599295,1549659295,2019-02-08 15:54:55,2019-12-26 03:01:04.727
1017,20190618.123037.5107311000,293064914,ELL04_402-1549599295_2_fixed.mat,2019/2/ELL04_402-1549599295_2_fixed.mat,2,ELL04_402-1549599295,1549599295,1549719295,2019-02-09 08:34:55,2019-12-26 03:01:04.727
1018,20190618.123102.4718909000,294326284,ELL04_402-1549599295_3_fixed.mat,2019/2/ELL04_402-1549599295_3_fixed.mat,3,ELL04_402-1549599295,1549599295,1549779295,2019-02-10 01:14:55,2019-12-26 03:01:04.727
1019,20190618.123038.0157845000,111062127,ELL04_402-1549599295_4_fixed.mat,2019/2/ELL04_402-1549599295_4_fixed.mat,4,ELL04_402-1549599295,1549599295,1549839295,2019-02-10 17:54:55,2019-12-26 03:01:04.727


In [56]:
df_files_mat[df_files_mat.fileID == 'ELL4_424-1445169540']

,CreationDTS,size,fileName,path,seg,fileID,unixStartTime,est_seg_start,segStartTime,CDACModifiedTime


In [57]:
df_pbm[df_pbm.fileID == 'BLK12_1252-1555973608']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
172,2019-04-27 11:24:00,2019-04-27 12:03:00,B1252,1.556382e+09,1.556385e+09,1.557179e+09,1.556077e+09,BLK12_1252-1555973608,1.555974e+09,1.557074e+09,2019-04-22 22:53:28,2792966,3248224202,B1252_1556382240,B1252_1556382240_1555973608
173,2019-04-30 12:14:00,2019-05-01 08:30:00,B1252,1.556644e+09,1.556717e+09,1.557179e+09,1.556077e+09,BLK12_1252-1555973608,1.555974e+09,1.557074e+09,2019-04-22 22:53:28,2792966,3248224202,B1252_1556644440,B1252_1556644440_1555973608
174,2019-04-27 15:31:00,2019-04-30 09:49:00,B1252,1.556397e+09,1.556636e+09,1.557179e+09,1.556077e+09,BLK12_1252-1555973608,1.555974e+09,1.557074e+09,2019-04-22 22:53:28,2792966,3248224202,B1252_1556397060,B1252_1556397060_1555973608
774,2019-04-26 18:20:00,2019-04-27 11:15:00,B1252,1.556321e+09,1.556382e+09,1.557179e+09,1.556077e+09,BLK12_1252-1555973608,1.555974e+09,1.557074e+09,2019-04-22 22:53:28,2792966,3248224202,B1252_1556320800,B1252_1556320800_1555973608
775,2019-05-01 10:37:00,2019-05-03 14:14:00,B1252,1.556725e+09,1.556911e+09,1.557179e+09,1.556077e+09,BLK12_1252-1555973608,1.555974e+09,1.557074e+09,2019-04-22 22:53:28,2792966,3248224202,B1252_1556725020,B1252_1556725020_1555973608


In [58]:
df_pbm[df_pbm.fileID == 'ELL04_406-1554232479']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
844,2019-04-02 15:00:00,2019-04-05 13:48:00,E406,1.554235e+09,1.554490e+09,1.555082e+09,1.554203e+09,ELL04_406-1554232479,1.554232e+09,1.554495e+09,2019-04-02 19:14:39,0684635,3235367593,E406_1554235200,E406_1554235200_1554232479


In [59]:
df_pbm[df_pbm.MRN == '6728191']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
917,2019-08-14 19:49:00,2019-08-18 18:23:00,B1254,1.565830e+09,1.566171e+09,1.566599e+09,1.565583e+09,BLK12_1254-1565826778,1.565827e+09,1.566195e+09,2019-08-14 23:52:58,6728191,3264388651,B1254_1565830140,B1254_1565830140_1565826778


In [60]:
df_pbm[df_pbm.MRN == '6674369']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [61]:
df_pbm[df_pbm.MRN == '6676303']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [62]:
df_pbm[df_pbm.MRN == '2021109']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
712,2019-03-19 04:05:00,2019-03-20 15:37:00,B1272,1.552986e+09,1.553114e+09,1.553201e+09,1.552977e+09,BLK12_1272-1552982319,1.552982e+09,1.553125e+09,2019-03-19 07:58:39,2021109,3242797367,B1272_1552986300,B1272_1552986300_1552982319


In [63]:
df_pbm[df_pbm.MRN == '021679']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [64]:
df_pbm[df_pbm.VisitIdentifier == '3264384407']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


#### Erik's macrocheck files:


In [65]:
df_pbm[df_pbm.VisitIdentifier == '3234926954']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [66]:
df_pbm[df_pbm.VisitIdentifier == '3237215185']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [67]:
df_pbm[df_pbm.VisitIdentifier == '3238224536']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [68]:
df_pbm[df_pbm.VisitIdentifier == '3241892606']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [69]:
df_pbm[df_pbm.VisitIdentifier == '3256787142']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [70]:
df_pbm[df_pbm.VisitIdentifier == '3260534406']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [71]:
df_pbm[df_pbm.VisitIdentifier == '3260589938']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [72]:
# df_pbm[df_pbm.VisitIdentifier == '']

In [73]:
# df_pbm[df_pbm.VisitIdentifier == '']

#### end of Erik's macrocheck files.




In [74]:
df_pbm[df_pbm.MRN == '0871462']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
575,2019-04-06 08:33:00,2019-04-11 19:02:00,B1274,1.554558e+09,1.555027e+09,1.555618e+09,1.554405e+09,BLK12_1274-1554554114,1.554554e+09,1.555046e+09,2019-04-06 12:35:14,0871462,3245471685,B1274_1554557580,B1274_1554557580_1554554114
918,2019-04-05 02:37:00,2019-04-05 15:06:00,B760,1.554450e+09,1.554495e+09,1.555618e+09,1.554405e+09,BLK07_760-1554446853,1.554447e+09,1.554525e+09,2019-04-05 06:47:33,0871462,3245471685,B760_1554449820,B760_1554449820_1554446853


In [75]:
ELL4 = np.unique([x for x in df_files_mat.fileID.values if 'ELL4' in x])
ELL04 = np.unique([x for x in df_files_mat.fileID.values if 'ELL04' in x])

In [76]:
ELL04

array(['ELL04_402-1546680576', 'ELL04_402-1546734000',
       'ELL04_402-1549599295', 'ELL04_402-1552424324',
       'ELL04_402-1553806081', 'ELL04_402-1554917998',
       'ELL04_404-1545529592', 'ELL04_404-1545577922',
       'ELL04_404-1558953113', 'ELL04_406-1526332156',
       'ELL04_406-1552416735', 'ELL04_406-1552681712',
       'ELL04_406-1552939168', 'ELL04_406-1552946447',
       'ELL04_406-1554232479', 'ELL04_408-1547506508',
       'ELL04_408-1547512952', 'ELL04_408-1550011150',
       'ELL04_408-1554754789', 'ELL04_410-1537717308',
       'ELL04_410-1550221696', 'ELL04_410-1550548798',
       'ELL04_410-1554296252', 'ELL04_412-1541393285',
       'ELL04_412-1541524281', 'ELL04_412-1548353408',
       'ELL04_412-1548630421', 'ELL04_412-1551138428',
       'ELL04_412-1552075373', 'ELL04_412-1557796108',
       'ELL04_414-1540833575', 'ELL04_414-1549673602',
       'ELL04_414-1549844349', 'ELL04_414-1560809185',
       'ELL04_416-1548558650', 'ELL04_416-1549482381',
       'EL

In [77]:
ELL4modified = [x.replace('ELL4','ELL04') for x in ELL4]
OnlyELL4 = set(ELL4modified) - set(ELL04)
# looks like those are not duplicates, i.e. only exist as ELL4 fileIDs but not ELL04.

In [78]:
df_pbm.head()

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
0,2019-07-03 09:19:00,2019-07-09 15:08:00,B1274,1.562164e+09,1.562703e+09,1.563044e+09,1.562135e+09,BLK12_1274-1561803974,1.561804e+09,1.562703e+09,2019-06-29 10:26:14,4736799,3258769573,B1274_1562163540,B1274_1562163540_1561803974
1,2019-07-03 09:19:00,2019-07-09 15:08:00,B1274,1.562164e+09,1.562703e+09,1.563044e+09,1.562135e+09,BLK12_1274-1562702709,1.562703e+09,1.562816e+09,2019-07-09 20:05:09,4736799,3258769573,B1274_1562163540,B1274_1562163540_1562702709
2,2019-06-23 10:33:00,2019-06-26 09:34:00,E402,1.561304e+09,1.561560e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1561303980,E402_1561303980_1561256864
4,2019-07-01 14:26:00,2019-07-06 17:08:00,E402,1.562009e+09,1.562451e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1562009160,E402_1562009160_1561256864
6,2019-06-26 13:41:00,2019-07-01 11:54:00,E402,1.561574e+09,1.562000e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1561574460,E402_1561574460_1561256864


In [79]:
df_files_mat[df_files_mat.fileID == 'ELL04_432-1528528595']

,CreationDTS,size,fileName,path,seg,fileID,unixStartTime,est_seg_start,segStartTime,CDACModifiedTime
483,20190618.122131.6010984000,390021266,ELL04_432-1528528595_0_fixed.mat,2018/6/ELL04_432-1528528595_0_fixed.mat,0,ELL04_432-1528528595,1528528595,1528528595,2018-06-09 03:16:35,2019-12-26 03:01:04.727
484,20190618.122119.8490735000,380613636,ELL04_432-1528528595_1_fixed.mat,2018/6/ELL04_432-1528528595_1_fixed.mat,1,ELL04_432-1528528595,1528528595,1528588595,2018-06-09 19:56:35,2019-12-26 03:01:04.727
485,20190618.122141.6398679000,401324967,ELL04_432-1528528595_2_fixed.mat,2018/6/ELL04_432-1528528595_2_fixed.mat,2,ELL04_432-1528528595,1528528595,1528648595,2018-06-10 12:36:35,2019-12-26 03:01:04.727
486,20190618.122104.4046864000,309151307,ELL04_432-1528528595_3_fixed.mat,2018/6/ELL04_432-1528528595_3_fixed.mat,3,ELL04_432-1528528595,1528528595,1528708595,2018-06-11 05:16:35,2019-12-26 03:01:04.727
487,20190618.122039.2862737000,313288716,ELL04_432-1528528595_4_fixed.mat,2018/6/ELL04_432-1528528595_4_fixed.mat,4,ELL04_432-1528528595,1528528595,1528768595,2018-06-11 21:56:35,2019-12-26 03:01:04.727
488,20190618.122104.3314645000,153539256,ELL04_432-1528528595_5_fixed.mat,2018/6/ELL04_432-1528528595_5_fixed.mat,5,ELL04_432-1528528595,1528528595,1528828595,2018-06-12 14:36:35,2019-12-26 03:01:04.727


In [80]:
table

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,ICU Site:,ICU Bed#:,MRN:,First Day Enrolled in Study:,First Name:,Last Name:
0,1,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,68,1298742.0,2018-06-06,Charles E,Sullivan
1,2,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Ellison 4,32,6378271.0,2018-06-11,Derek,Herling
2,3,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,54,1910753.0,2018-06-18,John,Carey
3,4,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 7,88,6188100.0,2018-06-19,Sheena,Foster
4,5,"ICU Day 1, am Screenng/Enroll",NaN,NaN,Blake 12,88,6051800.0,2018-06-19,Linda,Goodall
...,...,...,...,...,...,...,...,...,...,...
296,296,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,297,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,298,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,299,"ICU Day 1, am Screenng/Enroll",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:
import numpy as np
np.unique(df_pbm.MRN)

array(['0480086', '0684635', '0843227', '0871462', '1067855', '1104419',
       '1148346', '1194303', '1298742', '1403384', '1420433', '1469338',
       '1480537', '1486829', '1506383', '1649343', '1676863', '1723880',
       '1737814', '1742644', '1745708', '1750989', '1788840', '1810580',
       '1835922', '1845743', '1849492', '1876849', '1886786', '1902538',
       '1909198', '1910753', '1933581', '1933710', '1940438', '1975166',
       '1983461', '1990504', '1998902', '2010378', '2021109', '2086026',
       '2092575', '2149681', '2194847', '2215049', '2249421', '2264238',
       '2372026', '2380363', '2390500', '2466756', '2523280', '2595162',
       '2596463', '2597288', '2630974', '2647079', '2770155', '2771488',
       '2792966', '2853463', '2865943', '2881848', '2898315', '3042010',
       '3159260', '3165797', '3223814', '3243449', '3462701', '3468435',
       '3510348', '3591931', '3592022', '3603971', '3631517', '3708586',
       '3718149', '3724306', '3750433', '3759088', 

In [82]:
df_pbm.head()

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
0,2019-07-03 09:19:00,2019-07-09 15:08:00,B1274,1.562164e+09,1.562703e+09,1.563044e+09,1.562135e+09,BLK12_1274-1561803974,1.561804e+09,1.562703e+09,2019-06-29 10:26:14,4736799,3258769573,B1274_1562163540,B1274_1562163540_1561803974
1,2019-07-03 09:19:00,2019-07-09 15:08:00,B1274,1.562164e+09,1.562703e+09,1.563044e+09,1.562135e+09,BLK12_1274-1562702709,1.562703e+09,1.562816e+09,2019-07-09 20:05:09,4736799,3258769573,B1274_1562163540,B1274_1562163540_1562702709
2,2019-06-23 10:33:00,2019-06-26 09:34:00,E402,1.561304e+09,1.561560e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1561303980,E402_1561303980_1561256864
4,2019-07-01 14:26:00,2019-07-06 17:08:00,E402,1.562009e+09,1.562451e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1562009160,E402_1562009160_1561256864
6,2019-06-26 13:41:00,2019-07-01 11:54:00,E402,1.561574e+09,1.562000e+09,1.562627e+09,1.560536e+09,ELL04_402-1561256864,1.561257e+09,1.562448e+09,2019-06-23 02:27:44,2390500,3256083493,E402_1561574460,E402_1561574460_1561256864


In [83]:
df_pbm[df_pbm['MRN'] == '6663530']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid


In [84]:
# 	BLK10_1088-1557606617	

In [85]:
# BLK10_1058-1557612726	

In [86]:
df_pbm.shape

(650, 15)

In [87]:
len(set(df_pbm.fileID))

534

In [88]:
# save ALL fileID information to send to converion in other notebook

# not tested yet:
LogDir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/DataProcessingLog/'
# LogDir = 'C:/Users/wg984/Wolfgang/ICU-Sleep/'
filep=LogDir+'ToConvert.csv'

allfiles = pd.DataFrame(list(set(df_pbm.fileID)), columns = ['fileID'])
allfiles.to_csv(filep,index=False)

filep=LogDir+'ToConvertFullInfo.csv'
df_pbm.to_csv(filep,index=False)




In [89]:
allfiles.shape

(534, 1)

In [90]:
# studyID = table[table['MRN:']==int(ConversionStatus.iloc[rowId].MRN)]['Study ID:'].values[0]
# studyID

In [91]:
# ConversionStatus.iloc[rowId].studyID

In [92]:
# ConversionStatus['studyID'].iloc[rowId] = studyID.copy()

In [93]:
table.loc[table['Study ID:']=='31','MRN:'] = 1876849

In [94]:
# merge info about fileID and studyID

ConversionStatus = df_pbm.copy()
ConversionStatus['studyID'] = np.zeros([len(ConversionStatus),1],dtype=int)
ConversionStatus.head()

table = table.dropna(subset=['MRN:'])
table['MRN:'] = table['MRN:'].astype('int')


for rowId in range(len(ConversionStatus)):
    rowMRN = ConversionStatus.iloc[rowId].MRN
    if sum(table['MRN:']==int(ConversionStatus.iloc[rowId].MRN)) > 0:
        studyID = table[table['MRN:']==int(ConversionStatus.iloc[rowId].MRN)]['Study ID:'].values[0]
        if studyID == '98a': print('TEMP MRN 98a'); continue;
        ConversionStatus['studyID'].iloc[rowId] = int(studyID)

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\indexing.py:205: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


TEMP MRN 98a


In [95]:
print('TEMP MRN 98a')

ConversionStatus['studyID'] == '98a'

TEMP MRN 98a


C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\ops\__init__.py:1115: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = method(y)


0       False
1       False
2       False
4       False
6       False
        ...  
1024    False
1025    False
1026    False
1027    False
1028    False
Name: studyID, Length: 650, dtype: bool

In [96]:
cols = ConversionStatus.columns.tolist()
cols = cols[-1:] + cols[:-1]
cols = cols[:1] + cols[8:9] + cols[1:8] + cols[9:]
ConversionStatus = ConversionStatus[cols]


In [97]:
ConversionStatus.sort_values('studyID', inplace=True)
ConversionStatus.tail()

,studyID,fileID,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
729,192,BLK12_1260-1575747932,2019-12-07 14:35:00,2019-12-10 23:19:00,B1260,1.575747e+09,1.576038e+09,1.577855e+09,1.574766e+09,1.575748e+09,1.576055e+09,2019-12-07 19:45:32,1750989,3278402307,B1260_1575747300,B1260_1575747300_1575747932
730,192,BLK12_1272-1574812978,2019-11-26 22:31:00,2019-12-07 14:35:00,B1272,1.574825e+09,1.575747e+09,1.577855e+09,1.574766e+09,1.574813e+09,1.575789e+09,2019-11-27 00:02:58,1750989,3278402307,B1272_1574825460,B1272_1574825460_1574812978
727,192,BLK12_1260-1575747680,2019-12-07 14:35:00,2019-12-10 23:19:00,B1260,1.575747e+09,1.576038e+09,1.577855e+09,1.574766e+09,1.575748e+09,1.575748e+09,2019-12-07 19:41:20,1750989,3278402307,B1260_1575747300,B1260_1575747300_1575747680
541,193,BLK12_1252-1574537100,2019-11-23 14:54:00,2019-11-29 12:38:00,B1252,1.574539e+09,1.575049e+09,1.577468e+09,1.574180e+09,1.574537e+09,1.575277e+09,2019-11-23 19:25:00,4533154,3281186837,B1252_1574538840,B1252_1574538840_1574537100
920,194,BLK12_1290-1574722511,2019-11-25 18:31:00,2019-11-27 18:03:00,B1290,1.574725e+09,1.574896e+09,1.577140e+09,1.574688e+09,1.574723e+09,1.574966e+09,2019-11-25 22:55:11,6175987,3276933907,B1290_1574724660,B1290_1574724660_1574722511


In [98]:
filep=LogDir+'ConversionStatus_Step1.csv'
ConversionStatus.to_csv(filep,index=False)

In [99]:
ConversionStatus[ConversionStatus.fileID == 'BLK07_786-1554077017']

,studyID,fileID,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
276,35,BLK07_786-1554077017,2019-03-31 20:11:00,2019-04-08 10:40:00,B786,1.554081e+09,1.554738e+09,1.555779e+09,1.554081e+09,1.554077e+09,1.554740e+09,2019-04-01 00:03:37,6405518,3244689951,B786_1554081060,B786_1554081060_1554077017


In [100]:
df_pbm[df_pbm.fileID == 'BLK12_1288-1539370921']

,StartTime,EndTime,stpIn,tStart,tEnd,tDischarge,tAdmit,fileID,unixFileStartTime,unixFileEndTime,fileStartTime,MRN,VisitIdentifier,grp,ugid
476,2018-10-23 13:03:00,2018-10-25 10:09:00,B1288,1.540318e+09,1.540480e+09,1.543341e+09,1.539481e+09,BLK12_1288-1539370921,1.539371e+09,1.540819e+09,2018-10-12 19:02:01,6479611,3220820208,B1288_1540317780,B1288_1540317780_1539370921
814,2018-10-15 05:51:00,2018-10-16 15:39:00,B1288,1.539601e+09,1.539722e+09,1.543341e+09,1.539481e+09,BLK12_1288-1539370921,1.539371e+09,1.540819e+09,2018-10-12 19:02:01,6479611,3220820208,B1288_1539600660,B1288_1539600660_1539370921
816,2018-10-25 12:17:00,2018-10-28 18:00:00,B1288,1.540488e+09,1.540768e+09,1.543341e+09,1.539481e+09,BLK12_1288-1539370921,1.539371e+09,1.540819e+09,2018-10-12 19:02:01,6479611,3220820208,B1288_1540487820,B1288_1540487820_1539370921
817,2018-10-16 18:08:00,2018-10-23 09:01:00,B1288,1.539731e+09,1.540303e+09,1.543341e+09,1.539481e+09,BLK12_1288-1539370921,1.539371e+09,1.540819e+09,2018-10-12 19:02:01,6479611,3220820208,B1288_1539731280,B1288_1539731280_1539370921


In [101]:
# not converted:
not_converted = set(df_pbm['fileID']) - set(df_files_mat.fileID)
not_converted


{'BIG13_1324-1566393114',
 'BIG13_1324-1568995158',
 'BIG13_1326-1562086161',
 'BIG13_1346-1523492658',
 'BIG13_1346-1525106475',
 'BIG13_1348-1565290054',
 'BIG13_1352-1575497724',
 'BIG13_1354-1561066693',
 'BIG13_1354-1564758137',
 'BIG13_1354-1565033463',
 'BIG9_906-1566411801',
 'BIG9_936-1565191382',
 'BIG9_936-1571633766',
 'BIG9_938-1561236791',
 'BIG9_940-1573765531',
 'BIG9_940-1574204035',
 'BIG9_944-1563183492',
 'BIG9_944-1565033471',
 'BIG9_946-1566419282',
 'BLK07_752-1557712246',
 'BLK07_752-1557860944',
 'BLK07_752-1560992100',
 'BLK07_754-1568904993',
 'BLK07_756-1554043989',
 'BLK07_756-1555462981',
 'BLK07_756-1556049701',
 'BLK07_756-1564331542',
 'BLK07_756-1564519196',
 'BLK07_756-1574572874',
 'BLK07_756-1574741913',
 'BLK07_758-1571926490',
 'BLK07_760-1554446853',
 'BLK07_760-1565171101',
 'BLK07_760-1566429018',
 'BLK07_760-1566848858',
 'BLK07_760-1567622417',
 'BLK07_760-1567980569',
 'BLK07_760-1568178642',
 'BLK07_760-1568230344',
 'BLK07_766-1556104904',

In [57]:
# save unconverted fileID information to send to converion in other notebook

# not tested yet:
LogDir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/DataProcessingLog/'
filep=LogDir+'ToConvert.csv'

not_converted_df = pd.DataFrame(list(not_converted), columns = ['fileID'])
not_converted_df.to_csv(filep,index=False)
not_converted_df

PermissionError: [Errno 13] Permission denied: '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/DataProcessingLog/ToConvert.csv'

In [ ]:
for fileIDnotconverted in not_converted:
    print(df_pbm.loc[df_pbm['fileID'] == fileIDnotconverted,['StartTime','MRN']])

### Q: YP. those files are not converted. since i query for mrn, #i don't know if i need to convert them. can i just convert them anyway to be sure? or is it somehow possible to check on non converted files from which year they are and only if they match year (and maybe month) of icu sleep trial study, convert them.

In [73]:
### Get metadata if extracted
sql="""
select * from vw_BedmasterMetadata where
fileID in (%s)
""" % file_id_s

df_files_metadata=pd.read_sql_query(sql,cxn)

In [78]:
df_files_metadata.iloc[[1,5,44,200,444,555,777]]

,fileID,fileStartTime,tFileStartTime,location,seg,vs_metadata,wv_metadata,err,basename,CDACModifiedTime
1,BLK12_1288-1539370921,2018-10-12 19:02:01,1539370921,BLK12_1288,10,"[""HR"",""PVC"",""STII"",""STI"",""STIII"",""STAVR"",""STAV...","[""I"",""II"",""III"",""V"",""RR"",""SPO2""]",0,BLK12_1288-1539370921_10_fixed.mat,2019-04-21 15:40:46.077
5,BLK12_1288-1539370921,2018-10-12 19:02:01,1539370921,BLK12_1288,3,"[""HR"",""PVC"",""STII"",""STI"",""STV"",""STIII"",""STAVR""...","[""I"",""II"",""III"",""V"",""SPO2"",""AR1""]",0,BLK12_1288-1539370921_3_fixed.mat,2019-04-21 15:40:46.077
44,BLK12_1284-1540337896,2018-10-23 23:38:16,1540337896,BLK12_1284,3,"[""HR"",""PVC"",""STII"",""STI"",""STV"",""STIII"",""STAVR""...","[""I"",""II"",""III"",""V"",""SPO2"",""AR1"",""RR"",""AR2""]",0,BLK12_1284-1540337896_3_fixed.mat,2019-04-21 15:40:46.077
200,LUN08_814-1537794075,2018-09-24 13:01:15,1537794075,LUN08_814,5,"[""HR"",""PVC"",""STII"",""STI"",""STV"",""STIII"",""STAVR""...","[""I"",""II"",""III"",""V"",""SPO2""]",0,LUN08_814-1537794075_5_fixed.mat,2019-04-21 15:40:46.077
444,BLK08_876-1543782970,2018-12-02 20:36:10,1543782970,BLK08_876,1,[],[],0,BLK08_876-1543782970_1_fixed.mat,2019-05-20 06:01:03.203
555,BLK12_1260-1533560479,2018-08-06 13:01:19,1533560479,BLK12_1260,7,"[""STAVL"",""STV"",""AR1M"",""AR1R"",""AR1D"",""SPO2R"",""S...","[""SPO2"",""V"",""I"",""RR"",""II"",""AR1"",""III""]",0,BLK12_1260-1533560479_7_fixed.mat,2019-05-20 06:01:03.203
777,LUN07_788-1538084559,None,None,None,None,None,None,None,None,NaT


In [75]:
df_files_metadata.shape

(837, 10)

In [ ]:
print('Metadata extracted','|','filesID found')
print(len(df_files_metadata.groupby('fileID')),'|',len(set(df_pbm['fileID'])))

In [ ]:
# Get the location of files
sql="""
select * from vw_BedmasterFiles where
fileID in (%s)

""" % file_id_s

df_files=pd.read_sql_query(sql,cxn)

In [ ]:
# MRN, VisitId = CSN

# StartTime - ADT start time
# Endime - ADT end time

# tStart, tEnd, tAdmit, tDischarge - unix representation

# unixFileStartTime, unixFileEndTime - unix representation of the file start time and end time

df_files.head()

In [ ]:
df_files.archive_path[3]

In [ ]:
import os
pathN = os.path.dirname(df_files.archive_path[3]).split('final/')[1] + '/'
pathN

In [ ]:
df_files.loc[3]

In [ ]:
BM_MergedInfo = df_files_metadata.loc[df_files_metadata.fileID == 'ELL04_410-1537717308'].copy()

In [ ]:
import numpy as np

In [ ]:
BM_MergedInfo['path'] = BM_MergedInfo['basename'].apply(lambda x: pathN + x) 

In [ ]:
BM_MergedInfo

In [ ]:
# Output the metdata
OUTDIR='//mad3/MGH-NEURO-CDAC/Projects/Wolfgang/data'


df_pbm.to_csv(OUTDIR+'/vw_patientbedmaster.csv',index=False)

df_files.to_csv(OUTDIR+'/vw_bedmasterfiles.csv',index=False)

df_files_mat.to_csv(OUTDIR+'/vw_bedmastermat.csv',index=False)

df_files_metadata.to_csv(OUTDIR+'/vw_bedmastermetadata.csv',index=False)